# 8 Rainhas — Formulação Reduzida (Incremental)

Aplica as restrições do problema durante a geração de sucessores, reduzindo o espaço de busca de 1,8 x 10^14 para 2.057 estados.

**Técnica:** Busca incremental com poda  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/06-8-rainhas-formulacao-reduzida.ipynb)


In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║       PROBLEMA DAS 8 RAINHAS — FORMULAÇÃO REDUZIDA (INCREMENTAL)            ║
║                   Disciplina: Inteligência Artificial                        ║
║        Referência: Aula 4, Slides 33-34 — Russell & Norvig Cap. 3-6         ║
╚══════════════════════════════════════════════════════════════════════════════╝

VISÃO GERAL:
────────────
Este projeto implementa o problema das 8 Rainhas usando a Formulação Reduzida
(Incremental Melhorada), que reduz o espaço de busca de 1,8 × 10¹⁴ para apenas
2.057 estados, aplicando restrições DURANTE a geração de sucessores.

CONCEITOS FUNDAMENTAIS:
━━━━━━━━━━━━━━━━━━━━━━━

► ESPAÇO DE ESTADOS:
  Conjunto de TODOS os estados alcançáveis a partir do estado inicial por
  qualquer sequência de ações. Pode ser representado como um grafo onde
  nós são estados e arcos são ações (transições).

► ÁRVORE DE BUSCA:
  Estrutura gerada durante a exploração do espaço de estados. Diferente do
  espaço de estados (um grafo), a árvore de busca pode repetir estados
  (via caminhos distintos). A árvore de busca "expande sobre" o grafo.

► BUSCA INCREMENTAL:
  Em vez de partir de um estado completo e modificá-lo (busca local),
  a busca incremental CONSTRÓI a solução passo a passo, adicionando
  uma rainha por vez e verificando restrições em cada etapa.

► PODA IMPLÍCITA:
  Ao gerar APENAS sucessores válidos (sem conflitos), nunca colocamos
  estados inválidos na árvore de busca. Isso poda ramos inteiros antes
  mesmo de explorá-los — muito mais eficiente que gerar tudo e filtrar.

► ABSTRAÇÃO EM IA:
  A escolha de como REPRESENTAR o problema determina o tamanho do espaço
  de busca. A formulação reduzida é uma abstração que elimina simetrias e
  estados inválidos, mantendo apenas o essencial para encontrar a solução.

COMPARAÇÃO DE ESPAÇOS:
━━━━━━━━━━━━━━━━━━━━━━
  Formulação Clássica:  64×63×...×57 ≈ 1,8 × 10¹⁴ sequências
  Formulação Reduzida:  ≈ 2.057 estados válidos
  Redução:              fator > 86 bilhões de vezes menor!
"""

import time
import random
from typing import Optional


# ══════════════════════════════════════════════════════════════════════════════
# REPRESENTAÇÃO DO ESTADO
# ══════════════════════════════════════════════════════════════════════════════

"""
REPRESENTAÇÃO ESCOLHIDA: lista/vetor indexado por coluna.

  estado[coluna] = linha da rainha naquela coluna

Exemplo: [0, 4, 7, 5] representa um tabuleiro com 4 rainhas:
  - Coluna 0 → linha 0  (rainha em (0,0))
  - Coluna 1 → linha 4  (rainha em (4,1))
  - Coluna 2 → linha 7  (rainha em (7,2))
  - Coluna 3 → linha 5  (rainha em (5,3))
  - Colunas 4–7 → vazias (não preenchidas ainda)

POR QUE ESTA REPRESENTAÇÃO É EFICIENTE?
─────────────────────────────────────────
1. Índice = coluna: garante automaticamente UMA rainha por coluna.
   Conflitos de coluna são estruturalmente impossíveis.

2. Tamanho do vetor = número de rainhas posicionadas (0 a 8).
   O estado cresce incrementalmente — sem espaço desperdiçado.

3. Verificação de conflito em O(n): basta iterar pelas rainhas já
   posicionadas e checar linha e diagonal com a nova posição.

4. Reconstrução trivial: o vetor final É a solução.
"""


# ══════════════════════════════════════════════════════════════════════════════
# CLASSE PRINCIPAL — FORMULAÇÃO REDUZIDA
# ══════════════════════════════════════════════════════════════════════════════

class ProblemaOitoRainhasReduzido:
    """
    Formulação Reduzida (Incremental) do Problema das 8 Rainhas.

    ╔══════════════════════════════════════════════════════════╗
    ║            DEFINIÇÃO FORMAL DO PROBLEMA                  ║
    ╠══════════════════════════════════════════════════════════╣
    ║ Estados:    n rainhas (0≤n≤8), uma por coluna,          ║
    ║             nas n colunas mais à esquerda,               ║
    ║             sem nenhum par se atacando.                  ║
    ║                                                          ║
    ║ Inicial:    Tabuleiro vazio (n=0, estado=[])             ║
    ║                                                          ║
    ║ Sucessor:   Posiciona rainha na próxima coluna livre,   ║
    ║             apenas em linhas sem conflito.               ║
    ║                                                          ║
    ║ Objetivo:   n = 8  (8 rainhas posicionadas sem conflito) ║
    ╚══════════════════════════════════════════════════════════╝

    A CHAVE DA EFICIÊNCIA:
    ──────────────────────
    A função sucessor gera APENAS estados válidos.
    Estados com conflitos nunca são criados → poda implícita total.
    """

    def __init__(self, n: int = 8):
        self.n = n  # dimensão do tabuleiro

    # ─────────────────────────────────────────────────────────────
    # ESTADO INICIAL
    # ─────────────────────────────────────────────────────────────

    def estado_inicial(self) -> list:
        """
        Retorna o estado inicial: tabuleiro completamente vazio.

        Representado como lista vazia — nenhuma coluna preenchida.
        A busca parte daqui e constrói a solução incrementalmente.
        """
        return []

    # ─────────────────────────────────────────────────────────────
    # VERIFICAÇÃO DE CONFLITO
    # ─────────────────────────────────────────────────────────────

    def posicao_segura(self, estado: list, nova_linha: int) -> bool:
        """
        Verifica se posicionar uma rainha na próxima coluna, na linha
        'nova_linha', gera algum conflito com as rainhas já colocadas.

        A próxima coluna é implicitamente len(estado).

        CONFLITOS POSSÍVEIS:
        ─────────────────────
        1. Linha: outra rainha já ocupa 'nova_linha'
           → estado[c] == nova_linha

        2. Diagonal principal (↘): diferença de linhas == diferença de cols
           → nova_linha - estado[c] == nova_col - c
           → nova_linha - estado[c] > 0, subindo à direita

        3. Diagonal secundária (↗): soma de linha+col idêntica não é
           verificada separadamente — a condição |Δlinha| == |Δcol|
           cobre AMBAS as diagonais de forma unificada.

        VERIFICAÇÃO UNIFICADA:
          |nova_linha - estado[c]| == |nova_col - c|

        Complexidade: O(k) onde k = número de rainhas já posicionadas.

        Nota: conflitos de COLUNA são impossíveis por construção
        (cada índice do vetor representa uma coluna distinta).
        """
        nova_col = len(estado)  # próxima coluna a ser preenchida

        for coluna, linha in enumerate(estado):
            # Conflito de linha
            if linha == nova_linha:
                return False
            # Conflito diagonal (principal ou secundária)
            if abs(linha - nova_linha) == abs(coluna - nova_col):
                return False

        return True  # sem conflito: posição segura

    # ─────────────────────────────────────────────────────────────
    # FUNÇÃO SUCESSOR
    # ─────────────────────────────────────────────────────────────

    def gerar_sucessores(self, estado: list) -> list:
        """
        Função Sucessor da Formulação Reduzida.

        Identifica a próxima coluna vazia (len(estado)) e tenta
        posicionar uma rainha em cada linha (0 a n-1) dessa coluna.

        APENAS posições sem conflito geram sucessores.

        FATOR DE RAMIFICAÇÃO REDUZIDO:
        ────────────────────────────────
        Na formulação clássica: b ≈ 64, 63, ..., 57 por nível.
        Na formulação reduzida: b ≤ n (linhas válidas disponíveis).
        Na prática, conforme mais rainhas são colocadas, o fator cai
        drasticamente — muitas linhas ficam bloqueadas por conflitos.

        Isso explica a redução de 1,8 × 10¹⁴ para ~2.057 estados.

        Retorna lista de estados válidos (cada um = estado + nova rainha).
        """
        # Estado completo: não gera mais sucessores
        if len(estado) >= self.n:
            return []

        sucessores = []
        for linha in range(self.n):
            if self.posicao_segura(estado, linha):
                # Cria novo estado adicionando rainha na próxima coluna
                novo_estado = estado + [linha]
                sucessores.append(novo_estado)

        return sucessores

    # ─────────────────────────────────────────────────────────────
    # TESTE DE OBJETIVO
    # ─────────────────────────────────────────────────────────────

    def teste_objetivo(self, estado: list) -> bool:
        """
        Teste de Objetivo — verificação simplificada.

        Na formulação reduzida, um estado é solução quando e somente
        quando contém 8 rainhas, pois:

          ► Conflitos de coluna: impossíveis por representação
          ► Conflitos de linha e diagonal: impedidos pela função sucessor

        Portanto, basta verificar len(estado) == n.
        Não é necessário revalidar os conflitos!

        Esta simplificação é possível graças à PODA IMPLÍCITA:
        estados inválidos nunca são gerados.
        """
        return len(estado) == self.n

    # ─────────────────────────────────────────────────────────────
    # RENDERIZAÇÃO DO TABULEIRO
    # ─────────────────────────────────────────────────────────────

    def renderizar(self, estado: list) -> str:
        """
        Renderiza o tabuleiro visualmente.

        Saída:
          Q para rainha
          . para casa vazia

        Exemplo para [0, 4, 7, 5, 2, 6, 1, 3]:
          Q . . . . . . .
          . . . . . . Q .
          . . . . Q . . .
          . . . . . . . Q
          . Q . . . . . .
          . . . Q . . . .
          . . . . . . . .  ← coluna 6, linha 1 (Q na posição (1,6))
          . . Q . . . . .

        Complexidade: O(n²)
        """
        grid = [['.' for _ in range(self.n)] for _ in range(self.n)]
        for coluna, linha in enumerate(estado):
            grid[linha][coluna] = 'Q'
        return '\n'.join('  ' + ' '.join(row) for row in grid)

    # ─────────────────────────────────────────────────────────────
    # CÁLCULO DO ESPAÇO DE ESTADOS
    # ─────────────────────────────────────────────────────────────

    def sequencias_formulacao_classica(self) -> int:
        """Calcula 64 × 63 × ... × 57 (formulação clássica)."""
        total = 1
        for i in range(self.n * self.n, self.n * self.n - self.n, -1):
            total *= i
        return total


# ══════════════════════════════════════════════════════════════════════════════
# ESTRUTURA DO NÓ DA ÁRVORE DE BUSCA
# ══════════════════════════════════════════════════════════════════════════════

class No:
    """
    Nó da Árvore de Busca.

    CONCEITO:
    ─────────
    A árvore de busca é gerada DURANTE a exploração. Cada nó contém:
      - estado:        configuração atual (vetor de rainhas)
      - pai:           nó que gerou este nó (None se for a raiz)
      - acao:          qual rainha foi adicionada (linha, coluna)
      - profundidade:  nível na árvore = número de rainhas posicionadas
      - custo:         g(n) — aqui, cada ação tem custo 1

    DIFERENÇA NÓ × ESTADO:
    ──────────────────────
    Estado = configuração física do tabuleiro (o vetor)
    Nó     = estrutura de dados da árvore (contém o estado + metadados)

    Na busca incremental, profundidade == número de rainhas posicionadas.
    """

    def __init__(
        self,
        estado: list,
        pai: Optional['No'] = None,
        acao: Optional[tuple] = None,
        profundidade: int = 0,
        custo: int = 0
    ):
        self.estado = estado
        self.pai = pai
        self.acao = acao            # (linha, coluna) da rainha adicionada
        self.profundidade = profundidade
        self.custo = custo

    def __repr__(self):
        return (f"No(rainhas={len(self.estado)}, "
                f"prof={self.profundidade}, estado={self.estado})")


# ══════════════════════════════════════════════════════════════════════════════
# ALGORITMO DFS — BUSCA EM PROFUNDIDADE COM BACKTRACKING
# ══════════════════════════════════════════════════════════════════════════════

class BuscaDFS:
    """
    Busca em Profundidade (DFS) com Backtracking.

    POR QUE DFS É O ALGORITMO IDEAL PARA ESTE PROBLEMA?
    ─────────────────────────────────────────────────────
    1. MEMÓRIA BAIXA:
       DFS armazena apenas o caminho atual (profundidade = 8).
       Complexidade de espaço: O(d) onde d=8 — O(1) na prática!
       BFS armazenaria TODA a borda: impossível para problemas grandes.

    2. PROFUNDIDADE PEQUENA:
       A solução sempre está na profundidade 8 (exatamente 8 rainhas).
       DFS chega lá diretamente, sem explorar níveis desnecessários.

    3. ESPAÇO REDUZIDO:
       Com apenas ~2.057 estados válidos, DFS os percorre rapidamente.

    4. ADEQUADO PARA CSPs:
       Problemas de Satisfação de Restrições (CSPs) são naturalmente
       resolvidos por DFS + backtracking, pois:
       - A solução tem profundidade fixa (n variáveis = n rainhas)
       - Falhas são detectadas cedo (poda implícita)
       - Backtracking desfaz a última decisão e tenta outra

    BACKTRACKING:
    ─────────────
    Quando não há mais sucessores válidos em um estado (sem linha livre
    na próxima coluna), o algoritmo volta ao estado pai e tenta a próxima
    opção. Isso é naturalmente implementado pela pilha do DFS.

    COMPLEXIDADE:
    ─────────────
    • Tempo:   O(|estados válidos|) ≈ O(2.057) — extremamente eficiente
    • Espaço:  O(profundidade) = O(8) = O(1) para d fixo
    """

    def __init__(self, problema: ProblemaOitoRainhasReduzido):
        self.problema = problema
        self.estados_explorados = 0
        self.backtracks = 0

    def buscar(self, encontrar_todas: bool = False) -> list:
        """
        Executa a DFS a partir do estado inicial.

        Parâmetros:
          encontrar_todas: se True, encontra TODAS as soluções;
                           se False, retorna na primeira solução.

        Retorna lista de soluções (cada solução é um nó).

        FUNCIONAMENTO DO DFS:
        ──────────────────────
        1. Começa com pilha contendo apenas a raiz (estado vazio)
        2. Remove o topo da pilha (LIFO — Last In, First Out)
        3. Testa se é objetivo
        4. Gera sucessores válidos e empilha
        5. Repete até encontrar solução ou esvaziar a pilha

        O comportamento LIFO faz o DFS mergulhar profundamente antes
        de retroceder — exatamente o que queremos aqui.
        """
        solucoes = []

        # Estado inicial: tabuleiro vazio, profundidade 0
        no_raiz = No(
            estado=self.problema.estado_inicial(),
            pai=None,
            acao=None,
            profundidade=0,
            custo=0
        )

        # Pilha LIFO — o coração do DFS
        pilha = [no_raiz]

        while pilha:
            # Remove do topo (LIFO)
            no_atual = pilha.pop()
            self.estados_explorados += 1

            # TESTE DE OBJETIVO
            if self.problema.teste_objetivo(no_atual.estado):
                solucoes.append(no_atual)
                if not encontrar_todas:
                    return solucoes  # retorna imediatamente na 1ª solução
                continue

            # EXPANSÃO: gera sucessores válidos e empilha
            sucessores = self.problema.gerar_sucessores(no_atual.estado)

            if not sucessores and len(no_atual.estado) < self.problema.n:
                # Nenhum sucessor válido = beco sem saída → backtrack implícito
                self.backtracks += 1

            for novo_estado in reversed(sucessores):
                # reversed garante que a menor linha seja explorada primeiro
                coluna = len(no_atual.estado)
                linha = novo_estado[-1]
                no_filho = No(
                    estado=novo_estado,
                    pai=no_atual,
                    acao=(linha, coluna),
                    profundidade=no_atual.profundidade + 1,
                    custo=no_atual.custo + 1
                )
                pilha.append(no_filho)

        return solucoes

    def buscar_recursivo(
        self,
        estado: Optional[list] = None,
        profundidade: int = 0
    ) -> Optional[list]:
        """
        DFS Recursivo com Backtracking explícito.

        Versão alternativa mais intuitiva academicamente.
        Demonstra claramente o mecanismo de backtracking:
        quando a recursão retorna None, tentamos a próxima linha.

        BACKTRACKING EXPLÍCITO:
        ───────────────────────
        for linha in linhas_válidas:
            estado.append(linha)          ← tenta
            resultado = dfs(estado)
            if resultado: return resultado ← sucesso
            estado.pop()                  ← desfaz (backtrack)

        Esta versão usa a PILHA DE CHAMADAS do Python como pilha de busca,
        o que é elegante mas limitado pela profundidade de recursão.
        """
        if estado is None:
            estado = self.problema.estado_inicial()

        self.estados_explorados += 1

        # Caso base: objetivo atingido
        if self.problema.teste_objetivo(estado):
            return estado

        # Caso base: tabuleiro completo sem solução (não ocorre aqui,
        # pois o teste acima captura o caso de 8 rainhas)
        if len(estado) >= self.problema.n:
            return None

        # Tenta cada linha na próxima coluna
        for linha in range(self.problema.n):
            if self.problema.posicao_segura(estado, linha):
                # TENTA: adiciona a rainha
                novo_estado = estado + [linha]
                resultado = self.buscar_recursivo(novo_estado, profundidade + 1)
                if resultado is not None:
                    return resultado
                # BACKTRACK: tenta próxima linha (implícito no loop)
                self.backtracks += 1

        return None  # nenhuma linha válida → backtrack para o pai

    def reconstruir_caminho(self, no: No) -> list:
        """
        Reconstrói o caminho da raiz até a solução seguindo os ponteiros PAI.

        Retorna lista de (estado, ação) do estado inicial à solução.
        Permite visualizar cada passo da construção incremental.
        """
        caminho = []
        atual = no
        while atual is not None:
            caminho.append((atual.estado[:], atual.acao, atual.profundidade))
            atual = atual.pai
        caminho.reverse()
        return caminho


# ══════════════════════════════════════════════════════════════════════════════
# CONTADOR DE ESTADOS VÁLIDOS (para demonstração teórica)
# ══════════════════════════════════════════════════════════════════════════════

def contar_estados_validos(problema: ProblemaOitoRainhasReduzido) -> int:
    """
    Conta o número total de estados válidos na formulação reduzida.

    Este número é citado na literatura como aproximadamente 2.057.

    Usa BFS para contar todos os estados alcançáveis (não apenas soluções).
    """
    from collections import deque
    visitados = set()
    fila = deque([tuple(problema.estado_inicial())])
    visitados.add(tuple(problema.estado_inicial()))

    while fila:
        estado_tuple = fila.popleft()
        estado = list(estado_tuple)
        for suc in problema.gerar_sucessores(estado):
            suc_tuple = tuple(suc)
            if suc_tuple not in visitados:
                visitados.add(suc_tuple)
                fila.append(suc_tuple)

    return len(visitados)


# ══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES DE EXIBIÇÃO E RELATÓRIO
# ══════════════════════════════════════════════════════════════════════════════

def sep(char: str = '═', n: int = 70) -> str:
    return char * n

def titulo(texto: str, char: str = '═') -> str:
    pad = max(0, 70 - len(texto) - 4)
    return f"{char*(pad//2)}  {texto}  {char*(pad - pad//2)}"


def exibir_cabecalho():
    print()
    print(sep())
    print(titulo("PROBLEMA DAS 8 RAINHAS — FORMULAÇÃO REDUZIDA"))
    print(titulo("Busca Incremental com Restrições (DFS + Backtracking)", '─'))
    print(sep())
    print()


def exibir_teoria(problema: ProblemaOitoRainhasReduzido):
    seq_classica = problema.sequencias_formulacao_classica()
    estados_red = 2057

    print(titulo("FUNDAMENTAÇÃO TEÓRICA"))
    print()
    print("ESPAÇO DE ESTADOS:")
    print("  Conjunto de TODOS os estados alcançáveis a partir do estado")
    print("  inicial. Na formulação reduzida, inclui apenas estados VÁLIDOS")
    print("  (sem conflitos), pois a função sucessor faz poda implícita.")
    print()
    print("ÁRVORE DE BUSCA × ESPAÇO DE ESTADOS:")
    print("  O espaço de estados é um GRAFO (pode ter ciclos/múltiplos caminhos).")
    print("  A árvore de busca é gerada DURANTE a exploração — cada caminho")
    print("  da raiz a um nó representa uma sequência de ações distintas.")
    print("  Aqui, a árvore tem profundidade máxima 8 (uma rainha por nível).")
    print()
    print("PODA IMPLÍCITA:")
    print("  A função sucessor NÃO gera estados inválidos.")
    print("  Isso elimina ramos inteiros antes de explorá-los.")
    print("  É chamada de 'implícita' porque a poda ocorre na geração,")
    print("  não em um passo separado de verificação pós-geração.")
    print()
    print("FATOR DE RAMIFICAÇÃO:")
    print("  Formulação clássica: b ≈ 64 no nível 0 (qualquer quadrado)")
    print("  Formulação reduzida: b ≤ 8, tipicamente muito menor")
    print("  (linhas bloqueadas por conflitos reduzem as opções disponíveis)")
    print()
    print("ABSTRAÇÃO EM IA:")
    print("  A escolha da representação do problema determina diretamente")
    print("  o tamanho do espaço de busca. A formulação reduzida é uma")
    print("  abstração que elimina:")
    print("    • Estados com rainhas na mesma coluna (representação vetorial)")
    print("    • Estados com conflitos (verificação na geração de sucessores)")
    print("    • Permutações equivalentes (ordem de inserção fixa: esq→dir)")
    print()
    print(sep('─'))
    print("COMPARAÇÃO DE ESPAÇOS DE ESTADOS:")
    print(sep('─'))
    print()
    print(f"  Formulação Clássica:  {seq_classica:,}")
    print(f"                        ≈ 1,8 × 10¹⁴ sequências")
    print(f"  Formulação Reduzida:  ≈ {estados_red:,} estados válidos")
    print(f"  Redução:              fator ≈ {seq_classica // estados_red:,}×")
    print()
    print("  Por que a redução é tão drástica?")
    print("  1. Coluna fixa por índice: elimina n! permutações de colunas")
    print("  2. Ordem esquerda→direita: cada estado ocorre exatamente 1×")
    print("  3. Poda antecipada: conflitos são eliminados imediatamente")
    print("  4. Estados inválidos nunca entram na árvore de busca")
    print()
    print("  Resultado: o agente encontra a solução em microssegundos!")
    print()


def exibir_formulacao(problema: ProblemaOitoRainhasReduzido):
    print(titulo("FORMULAÇÃO DO PROBLEMA"))
    print()
    print("  ESTADOS:")
    print("    • n rainhas (0 ≤ n ≤ 8), uma por coluna")
    print("    • Nas n colunas mais à esquerda")
    print("    • Nenhum par de rainhas se ataca")
    print("    • Representação: lista onde índice=coluna, valor=linha")
    print()
    print("  ESTADO INICIAL:  [] (tabuleiro vazio, 0 rainhas)")
    print()
    print("  FUNÇÃO SUCESSOR:")
    print("    • Identifica a próxima coluna vazia (len(estado))")
    print("    • Tenta posicionar rainha em cada linha 0..7")
    print("    • Gera APENAS estados sem conflito")
    print("    • Conflitos verificados: linha, diagonal ↘, diagonal ↗")
    print()
    print("  TESTE DE OBJETIVO:")
    print("    • len(estado) == 8")
    print("    • (conflitos já garantidos ausentes pela função sucessor)")
    print()
    print("  EXEMPLO DE ESTADO PARCIAL [0, 4, 7, 5]:")

    exemplo = ProblemaOitoRainhasReduzido(8)
    grid = [['.' for _ in range(8)] for _ in range(8)]
    for col, lin in enumerate([0, 4, 7, 5]):
        grid[lin][col] = 'Q'
    for lin, row in enumerate(grid):
        marcador = " ←" if lin in [0, 4, 7, 5] else ""
        print(f"    {'  '.join(row)}{marcador}")
    print()
    print("    Colunas 0-3 preenchidas, colunas 4-7 vazias.")
    print("    Nenhuma rainha ataca outra neste estado parcial.")
    print()


def exibir_algoritmo():
    print(titulo("POR QUE DFS COM BACKTRACKING?"))
    print()
    print("  COMPARAÇÃO DE ALGORITMOS PARA ESTE PROBLEMA:")
    print()
    print("  BFS (Busca em Largura):")
    print("    • Complexidade de espaço: O(b^d) — armazena toda a borda")
    print("    • Para d=8 e b≈6: dezenas de milhares de nós em memória")
    print("    • Encontra solução de menor profundidade (desnecessário aqui)")
    print("    • Inadequado: solução sempre está em profundidade 8")
    print()
    print("  DFS (Busca em Profundidade) + Backtracking:")
    print("    • Complexidade de espaço: O(d) = O(8) ≈ O(1)")
    print("    • Armazena apenas o caminho atual na pilha")
    print("    • Backtracking: desfaz última decisão ao encontrar beco")
    print("    • IDEAL para CSPs com profundidade de solução fixa")
    print("    • Adequado para encontrar UMA solução rapidamente")
    print()
    print("  BUSCA LOCAL (Hill Climbing):")
    print("    • Parte de estado COMPLETO e modifica")
    print("    • Não constrói incrementalmente")
    print("    • Pode ficar em mínimos locais")
    print("    • Não garante encontrar solução")
    print()
    print("  CONCLUSÃO: DFS + backtracking + poda implícita")
    print("  é a combinação ótima para este problema.")
    print()


def exibir_resultado_dfs(
    no_solucao: No,
    busca: BuscaDFS,
    tempo: float,
    problema: ProblemaOitoRainhasReduzido
):
    print(titulo("RESULTADO DA BUSCA DFS"))
    print()
    print(f"  ✓ SOLUÇÃO ENCONTRADA!")
    print()
    print(f"  Métricas da busca:")
    print(f"  • Estados explorados:  {busca.estados_explorados}")
    print(f"  • Backtracks:          {busca.backtracks}")
    print(f"  • Profundidade:        {no_solucao.profundidade}")
    print(f"  • Custo do caminho:    {no_solucao.custo}")
    print(f"  • Tempo de execução:   {tempo*1000:.4f} ms")
    print()
    print(f"  Solução (vetor): {no_solucao.estado}")
    print()
    print("  TABULEIRO SOLUÇÃO:")
    print()
    print(problema.renderizar(no_solucao.estado))
    print()
    print("  Legenda por coluna:")
    for col, lin in enumerate(no_solucao.estado):
        print(f"    Coluna {col} → linha {lin}  (posição ({lin},{col}))")
    print()


def exibir_caminho(no_solucao: No, busca: BuscaDFS, problema: ProblemaOitoRainhasReduzido):
    print(titulo("CONSTRUÇÃO INCREMENTAL DA SOLUÇÃO"))
    print()
    print("  Cada passo adiciona uma rainha à próxima coluna vazia.")
    print("  Apenas posições sem conflito são consideradas.")
    print()

    caminho = busca.reconstruir_caminho(no_solucao)
    for estado, acao, prof in caminho:
        if acao is None:
            print(f"  Passo 0 | Estado inicial: [] (tabuleiro vazio)")
        else:
            lin, col = acao
            suc_disponiveis = len(problema.gerar_sucessores(
                estado[:-1] if estado else []
            ))
            print(f"  Passo {prof} | Coluna {col} → linha {lin} "
                  f"| Estado parcial: {estado}")
    print()


def exibir_todas_solucoes(total: int, tempo_todas: float):
    print(titulo("TODAS AS SOLUÇÕES"))
    print()
    print(f"  Total de soluções distintas encontradas: {total}")
    print(f"  (Valor clássico para n=8: 92 soluções)")
    print(f"  Tempo para encontrar todas: {tempo_todas*1000:.2f} ms")
    print()
    print("  As 92 soluções incluem rotações e reflexões.")
    print("  Soluções fundamentalmente distintas (sem simetria): 12")
    print()


def exibir_comparacao_final(problema: ProblemaOitoRainhasReduzido, estados_validos: int):
    seq_classica = problema.sequencias_formulacao_classico = problema.sequencias_formulacao_classica()

    print(titulo("COMPARAÇÃO CRÍTICA — IMPACTO DA FORMULAÇÃO"))
    print()
    print("┌──────────────────────────────────────────────────────────────┐")
    print("│            FORMULAÇÃO CLÁSSICA (estados completos)           │")
    print("├──────────────────────────────────────────────────────────────┤")
    print("│  Estado inicial: tabuleiro vazio                             │")
    print("│  Sucessor: rainha em QUALQUER quadrado livre                 │")
    print(f"│  Espaço: ≈ 1,8 × 10¹⁴ sequências ({seq_classica:,.0f})  │")
    print("│  Gera estados inválidos (conflitos permitidos)               │")
    print("│  BFS/DFS puros: inviáveis sem poda adicional                 │")
    print("│  Alto custo computacional e de memória                       │")
    print("└──────────────────────────────────────────────────────────────┘")
    print()
    print("┌──────────────────────────────────────────────────────────────┐")
    print("│       FORMULAÇÃO REDUZIDA (incremental com restrições)       │")
    print("├──────────────────────────────────────────────────────────────┤")
    print("│  Estado inicial: tabuleiro vazio                             │")
    print("│  Sucessor: rainha apenas em posições VÁLIDAS da próx. col.  │")
    print(f"│  Espaço: ≈ {estados_validos} estados válidos (confirmado por contagem)     │")
    print("│  Nunca gera estados inválidos — poda implícita total         │")
    print("│  DFS com backtracking: solução em microssegundos             │")
    print("│  Custo de memória: O(8) ≈ O(1)                              │")
    print("└──────────────────────────────────────────────────────────────┘")
    print()
    print("  LIÇÃO FUNDAMENTAL DE IA:")
    print("  ─────────────────────────")
    print("  A ESCOLHA DA FORMULAÇÃO DO PROBLEMA é tão importante quanto")
    print("  a escolha do algoritmo de busca.")
    print()
    print("  Uma boa formulação pode reduzir o espaço de busca em")
    print(f"  mais de {seq_classica // estados_validos:,} vezes — como demonstrado aqui.")
    print()
    print("  A formulação reduzida aplica o princípio de ABSTRAÇÃO EM IA:")
    print("  representar apenas o essencial, eliminando redundâncias e")
    print("  impossibilidades estruturais antes mesmo da busca começar.")
    print()


# ══════════════════════════════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def main():
    problema = ProblemaOitoRainhasReduzido(n=8)

    # ── Cabeçalho ──
    exibir_cabecalho()

    # ── Teoria ──
    exibir_teoria(problema)

    # ── Formulação ──
    exibir_formulacao(problema)

    # ── Por que DFS? ──
    exibir_algoritmo()

    # ── DFS: primeira solução ──
    print(titulo("EXECUTANDO DFS — PRIMEIRA SOLUÇÃO"))
    print()
    busca = BuscaDFS(problema)
    inicio = time.time()
    solucoes = busca.buscar(encontrar_todas=False)
    tempo = time.time() - inicio

    if solucoes:
        exibir_resultado_dfs(solucoes[0], busca, tempo, problema)
        exibir_caminho(solucoes[0], busca, problema)
    else:
        print("  ✗ Nenhuma solução encontrada.")
    print()

    # ── DFS recursivo (demonstração alternativa) ──
    print(titulo("DFS RECURSIVO COM BACKTRACKING EXPLÍCITO"))
    print()
    print("  Versão recursiva — demonstra o backtracking de forma didática:")
    print()
    busca_rec = BuscaDFS(problema)
    inicio_rec = time.time()
    solucao_rec = busca_rec.buscar_recursivo()
    tempo_rec = time.time() - inicio_rec

    if solucao_rec:
        print(f"  ✓ Solução: {solucao_rec}")
        print(f"  Estados explorados: {busca_rec.estados_explorados}")
        print(f"  Backtracks: {busca_rec.backtracks}")
        print(f"  Tempo: {tempo_rec*1000:.4f} ms")
        print()
        print("  TABULEIRO:")
        print()
        print(problema.renderizar(solucao_rec))
    print()

    # ── Todas as soluções ──
    print(titulo("ENCONTRANDO TODAS AS 92 SOLUÇÕES"))
    print()
    busca_todas = BuscaDFS(problema)
    inicio_todas = time.time()
    todas = busca_todas.buscar(encontrar_todas=True)
    tempo_todas = time.time() - inicio_todas
    exibir_todas_solucoes(len(todas), tempo_todas)

    # ── Contagem de estados válidos ──
    print(titulo("CONTAGEM DE ESTADOS VÁLIDOS (verificação)"))
    print()
    print("  Contando todos os estados alcançáveis na formulação reduzida...")
    inicio_cont = time.time()
    total_estados = contar_estados_validos(problema)
    tempo_cont = time.time() - inicio_cont
    print(f"  Total de estados válidos: {total_estados}")
    print(f"  Referência da literatura: ≈ 2.057")
    print(f"  Tempo de contagem: {tempo_cont*1000:.2f} ms")
    print()

    # ── Comparação final ──
    exibir_comparacao_final(problema, total_estados)

    # ── Resumo ──
    print(sep())
    print(titulo("RESUMO EXECUTIVO"))
    print(sep())
    print()
    print("  Formulação Clássica:")
    print(f"    → {problema.sequencias_formulacao_classica():,} sequências ≈ 1,8×10¹⁴")
    print()
    print("  Formulação Reduzida:")
    print(f"    → {total_estados} estados válidos")
    print(f"    → DFS encontrou solução em {tempo*1000:.4f} ms")
    print(f"    → 92 soluções encontradas em {tempo_todas*1000:.2f} ms")
    print()
    print("  A modelagem correta do problema é a diferença entre")
    print("  uma busca inviável e uma resolução em microssegundos.")
    print()
    print(sep())


if __name__ == "__main__":
    main()


══════════════════════════════════════════════════════════════════════
═══════════  PROBLEMA DAS 8 RAINHAS — FORMULAÇÃO REDUZIDA  ═══════════
──────  Busca Incremental com Restrições (DFS + Backtracking)  ───────
══════════════════════════════════════════════════════════════════════

══════════════════════  FUNDAMENTAÇÃO TEÓRICA  ═══════════════════════

ESPAÇO DE ESTADOS:
  Conjunto de TODOS os estados alcançáveis a partir do estado
  inicial. Na formulação reduzida, inclui apenas estados VÁLIDOS
  (sem conflitos), pois a função sucessor faz poda implícita.

ÁRVORE DE BUSCA × ESPAÇO DE ESTADOS:
  O espaço de estados é um GRAFO (pode ter ciclos/múltiplos caminhos).
  A árvore de busca é gerada DURANTE a exploração — cada caminho
  da raiz a um nó representa uma sequência de ações distintas.
  Aqui, a árvore tem profundidade máxima 8 (uma rainha por nível).

PODA IMPLÍCITA:
  A função sucessor NÃO gera estados inválidos.
  Isso elimina ramos inteiros antes de explorá-los.
  É chamada d